# AxonML Draft-Distill Training (Colab Pro / Vertex A100)

Distills Qwen3-1.7B teacher → Qwen3-0.6B student on FineWeb (~1B tokens seen,
30k steps × bs=16 × seq=2048). Wall-clock ~12-16h on A100-40GB, ~8-10h on A100-80GB.

**Before running:** Runtime → Change runtime type → A100 GPU. All cells idempotent.


In [ ]:
# 1. Environment + paths. Detects Colab vs Vertex automatically.
import os, sys, subprocess
IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    BASE = '/content'
elif os.path.isdir('/home/jupyter'):
    BASE = '/home/jupyter'
else:
    BASE = '/root'
print(f'IS_COLAB={IS_COLAB} BASE={BASE}')
%env BASE=$BASE
!nvidia-smi | head -4
!nvcc --version | head -4 || echo 'no nvcc yet — installing below'
!df -h $BASE 2>/dev/null
!free -h | head -2
!echo HOME=$HOME WHOAMI=$(whoami) PWD=$(pwd)


In [ ]:
# 2. Colab-only: mount Google Drive for persistent checkpoints across
# session disconnects. Vertex/local run skips this cell.
import os, sys
CKPT_BASE = f"{os.environ['BASE']}/checkpoints"
if 'google.colab' in sys.modules:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    drive_dir = '/content/drive/MyDrive/axonml_distill_checkpoints'
    os.makedirs(drive_dir, exist_ok=True)
    CKPT_BASE = drive_dir
    print(f'Using Drive for checkpoints: {CKPT_BASE}')
else:
    os.makedirs(CKPT_BASE, exist_ok=True)
    print(f'Using local for checkpoints: {CKPT_BASE}')
%env CKPT=$CKPT_BASE


In [ ]:
# 3. Install Rust 1.85 (matches AxonML rust-version)
import os, subprocess
if subprocess.run(['which', 'cargo'], capture_output=True).returncode != 0:
    os.system("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain 1.85 --profile minimal")
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:" + os.environ['PATH']
!rustc --version && cargo --version


In [ ]:
# 4. Clone AxonML + set up data/model dirs
import os
BASE = os.environ['BASE']
SRC = f'{BASE}/AxonML'
DATA = f'{BASE}/datasets'
MODELS = f'{BASE}/models'
os.makedirs(DATA, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)
%env SRC=$SRC
%env DATA=$DATA
%env MODELS=$MODELS
!if [ ! -d "$SRC/.git" ]; then rm -rf $SRC && git clone --depth 1 https://github.com/AutomataNexus/AxonML.git $SRC; else cd $SRC && git pull --ff-only; fi
!cd $SRC && git log --oneline -3


In [ ]:
# 5. Detect GPU architecture and recompile PTX for it. AxonML ships PTX
# compiled for sm_89 (Ada / RTX 5070 Ti laptop); Colab A100 is sm_80,
# T4 is sm_75, L4 is sm_89. sm_89 PTX can't JIT-lower to sm_80 so we
# recompile every .cu with the detected arch BEFORE cargo build
# (cargo include_str!'s the .ptx files).
import subprocess, re
out = subprocess.check_output(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader']).decode().strip()
cc = out.splitlines()[0].strip().replace('.', '')  # e.g. '8.0' -> '80'
arch = f'sm_{cc}'
print(f'GPU compute capability: {cc} -> arch={arch}')
%env ARCH=$arch
!cd $SRC/crates/axonml-core/src/backends/cuda_kernels && \
  for f in *.cu; do \
    base=$(basename "$f" .cu); \
    nvcc -ptx -arch=$ARCH --use_fast_math "$f" -o "${base}.ptx" || exit 1; \
  done && ls -lh *.ptx | head -15


In [ ]:
# 6. Build train_draft_distill + tokenize_corpus with CUDA. ~15 min first time.
import os
os.chdir(os.environ['SRC'])
!cargo build --release --features cuda -p llm-training --bin train_draft_distill --bin tokenize_corpus 2>&1 | tail -8
!ls -lh llm-training/target/release/train_draft_distill llm-training/target/release/tokenize_corpus


In [ ]:
# 7. Download teacher GGUF (Qwen3-1.7B) ~1.2 GB
import os
teacher_dir = os.path.join(os.environ['MODELS'], 'qwen3-1.7b')
teacher_path = os.path.join(teacher_dir, 'Qwen3-1.7B-Q4_K_M.gguf')
os.makedirs(teacher_dir, exist_ok=True)
%env TEACHER=$teacher_path
!if [ ! -s "$TEACHER" ]; then \
    wget -q --show-progress -O "$TEACHER" \
      https://huggingface.co/Qwen/Qwen3-1.7B-GGUF/resolve/main/Qwen3-1.7B-Q4_K_M.gguf; \
  fi
!ls -lh "$TEACHER"


In [ ]:
# 8. Download FineWeb sample-10BT shard (~2 GB parquet)
import os
fw_dir = os.path.join(os.environ['DATA'], 'fineweb')
os.makedirs(os.path.join(fw_dir, 'raw'), exist_ok=True)
parquet_path = os.path.join(fw_dir, 'raw', '000.parquet')
%env PARQUET=$parquet_path
!if [ ! -s "$PARQUET" ]; then \
    wget -q --show-progress -O "$PARQUET" \
      https://huggingface.co/datasets/HuggingFaceFW/fineweb/resolve/main/sample/10BT/000_00000.parquet; \
  fi
!ls -lh "$PARQUET"


In [ ]:
# 9. Parquet -> flat text. FineWeb has a 'text' column.
import pyarrow.parquet as pq
from pathlib import Path
import os

parquet_path = Path(os.environ['PARQUET'])
text_path = Path(os.environ['DATA']) / 'fineweb' / 'fineweb.txt'
text_path.parent.mkdir(parents=True, exist_ok=True)

if text_path.exists() and text_path.stat().st_size > 1_000_000_000:
    print(f'text already exists: {text_path} ({text_path.stat().st_size/1e9:.2f} GB) — skip')
else:
    pf = pq.ParquetFile(parquet_path)
    print(f'parquet rows: {pf.metadata.num_rows}  columns: {pf.schema.names}')
    total_chars = 0; total_rows = 0
    with open(text_path, 'w', encoding='utf-8') as f:
        for batch in pf.iter_batches(batch_size=20000, columns=['text']):
            for t in batch.column('text').to_pylist():
                if t:
                    f.write(t); f.write('\n\n')
                    total_chars += len(t) + 2; total_rows += 1
    print(f'wrote {total_rows:,} rows / {total_chars/1e9:.2f} GB to {text_path}')


In [ ]:
# 10. Tokenize with Qwen3 BPE -> flat uint32 stream (~10 min for 2 GB text)
import os
tokens_bin = os.path.join(os.environ['DATA'], 'fineweb', 'fineweb.qwen3.bin')
%env TOKENS=$tokens_bin
!if [ ! -s "$TOKENS" ]; then \
    $SRC/llm-training/target/release/tokenize_corpus \
      --gguf $TEACHER \
      --input $DATA/fineweb/fineweb.txt \
      --output $TOKENS; \
  fi
!ls -lh $DATA/fineweb/


In [ ]:
# 11. Launch training. 30k steps × bs=16 × seq=2048 ≈ 1B tokens seen.
# Expected A100-40GB: ~12-16h | A100-80GB: ~8-10h | L4: ~40h (won't fit in one session).
# Checkpoints every 1k steps, keep last 5.
# --resume latest = safe to re-run this cell if Colab disconnects mid-run.
import os, time
output_dir = os.path.join(os.environ['CKPT'], 'draft_distill')
os.makedirs(output_dir, exist_ok=True)
%env OUTPUT_DIR=$output_dir
%env LOG_PATH=/tmp/distill.log
print(f'launching at {time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'checkpoints -> {output_dir}')
!cd $SRC && \
  ./llm-training/target/release/train_draft_distill \
    --teacher-gguf $TEACHER \
    --tokens-bin $TOKENS \
    --arch 0.6b \
    --seq-len 2048 \
    --batch-size 16 \
    --epochs 1 \
    --steps 30000 \
    --lr 3e-4 \
    --warmup 500 \
    --temperature 3.0 \
    --ce-weight 0.1 \
    --weight-decay 0.1 \
    --grad-clip 1.0 \
    --log-every 50 \
    --generate-every 0 \
    --checkpoint-every-steps 1000 \
    --keep-last-k 5 \
    --output-dir $OUTPUT_DIR \
    --resume latest \
    2>&1 | tee $LOG_PATH
print(f'training exit at {time.strftime("%Y-%m-%d %H:%M:%S")}')


In [ ]:
# 12. Done — stage results for retrieval.
# On Colab: checkpoints already live on Google Drive (persisted at /content/drive/MyDrive/axonml_distill_checkpoints/draft_distill/).
# On Vertex: upload to GCS bucket.
import sys, time, os
stamp = time.strftime('%Y%m%dT%H%M%S')
if 'google.colab' in sys.modules:
    print(f'Checkpoints persisted on Google Drive at:')
    print(f'  {os.environ["OUTPUT_DIR"]}')
    print(f'Pull from local with: rclone copy / gcloud auth / or just sync via Drive desktop client.')
    !cp $LOG_PATH $OUTPUT_DIR/distill.log
    !ls -lh $OUTPUT_DIR/
else:
    gcs_dst = f'gs://your-gcs-bucket/runs/distill-{stamp}'
    %env GCS_DST=$gcs_dst
    !gsutil -m cp -r $OUTPUT_DIR $GCS_DST/
    !gsutil cp $LOG_PATH $GCS_DST/distill.log
    print(f'done. pull with:\n  gsutil -m cp -r {gcs_dst} /opt/AxonML/checkpoints/draft_distill/')
